# Sliding Window — Part 2: The Two Types

This is where most people get confused. There are **two completely different problems** that look similar but need different strategies.

---

## 🟢 Type 1: Find Longest / Shortest

> "What is the **length** of the longest/shortest subarray that satisfies condition X?"

**ONE answer** — you track a single best window.

Examples:
- Longest substring without repeating characters
- Longest subarray with at most k zeros
- Longest substring with at most 2 distinct chars

---

## 🔴 Type 2: Count How Many Subarrays
> "How **many** subarrays satisfy condition X?"

**MANY answers** — for each right pointer, multiple left boundaries can be valid.

Examples:
- Number of subarrays with sum = k
- Number of subarrays with exactly k distinct chars
- Number of subarrays with product < k

---

## Why Does This Matter?

```
arr = [1, 0, 1, 0, 1]   goal = 2

At r=4 (last 1), valid windows ending here:
  [1, 0, 1, 0, 1]  sum=3  ❌
  [0, 1, 0, 1]     sum=2  ✅
  [1, 0, 1]        sum=2  ✅
  [0, 1]           sum=1  ❌
  [1]              sum=1  ❌

→ 2 valid windows end at r=4 alone!
```

A single `l` pointer can't capture all of them — you'd need to count ALL valid left positions.

---
## Type 1 Deep Dive: Longest Substring Without Repeating Characters

In [ ]:
def length_of_longest_substring(s):
    """
    Type 1 — find LONGEST. One answer, one window.
    
    Window invalid when: duplicate character enters.
    Fix: jump l past the previous occurrence of the duplicate.
    """
    seen = {}  # char -> most recent index
    l    = 0
    best = 0

    for r in range(len(s)):
        if s[r] in seen:
            l = max(l, seen[s[r]] + 1)   # jump past previous occurrence
                                          # max() prevents l from going backward
        seen[s[r]] = r
        best = max(best, r - l + 1)

    return best


print(length_of_longest_substring("abcabcbb"))  # 3
print(length_of_longest_substring("pwwkew"))    # 3
print(length_of_longest_substring("bbbbb"))     # 1

```
s = "abcabcbb"

r=0 a: window=[a]         l=0  len=1
r=1 b: window=[a,b]       l=0  len=2
r=2 c: window=[a,b,c]     l=0  len=3  ← best
r=3 a: duplicate! l→1     window=[b,c,a]  l=1  len=3
r=4 b: duplicate! l→2     window=[c,a,b]  l=2  len=3
r=5 c: duplicate! l→3     window=[a,b,c]  l=3  len=3
r=6 b: duplicate! l→5     window=[c,b]    l=5  len=2
r=7 b: duplicate! l→7     window=[b]      l=7  len=1
```

---
## Type 1 Deep Dive: Max Consecutive Ones III (at most k flips)

In [ ]:
def longest_ones(nums, k):
    """
    Type 1 — find LONGEST.
    
    Window invalid when: zero_count > k.
    
    Optimized trick: never SHRINK the window below the best size found.
    When invalid, just slide l forward by 1 instead of while-looping.
    This works because we only care about finding a BIGGER window.
    """
    l = 0

    for r in range(len(nums)):
        if nums[r] == 0:
            k -= 1               # used a flip

        if k < 0:                # window invalid: too many zeros
            if nums[l] == 0:
                k += 1           # reclaim flip as left edge leaves
            l += 1               # slide, don't shrink

    return len(nums) - l         # window never shrinks, final size = max size


print(longest_ones([1,1,1,0,0,0,1,1,1,1,0], 2))               # 6
print(longest_ones([0,0,1,1,0,0,1,1,1,0,1,1,0,0,0,1,1,1,1], 3))  # 10

---
## Type 2 Deep Dive: Subarray Sum Equals K (prefix sum approach)

### Why sliding window FAILS here

```python
# Attempted sliding window — WRONG for this problem
for r in range(n):
    total += nums[r]
    while total > goal:   # ← This only works if all values are POSITIVE
        total -= nums[l]
        l += 1
    if total == goal:
        count += 1         # ← Misses windows: only counts one l per r
```

Two problems:
1. If array has zeros (like `[1,0,1,0,1]`), shrinking past a zero doesn't change the sum — window gets stuck
2. For each `r`, there may be **multiple valid** `l` positions — a single `l` misses them

### The prefix sum fix

```
prefix[r] - prefix[l-1] = goal
→  prefix[l-1] = prefix[r] - goal

So: for each r, count how many PAST prefix sums equal (current_prefix - goal)
```

In [ ]:
def num_subarrays_with_sum(nums, goal):
    """
    Type 2 — COUNT subarrays.
    
    For each r, we want: how many l's exist where prefix[r] - prefix[l] = goal?
    i.e. how many times has the value (prefix[r] - goal) appeared before?
    
    prefix = {0: 1} seeds the map — handles subarrays starting at index 0
    (an empty prefix before the array has sum 0, and it appears once).
    """
    total  = 0
    count  = 0
    prefix = {0: 1}   # prefix_sum -> how many times seen

    for r in range(len(nums)):
        total += nums[r]

        # How many left boundaries give us exactly 'goal'?
        count += prefix.get(total - goal, 0)

        prefix[total] = 1 + prefix.get(total, 0)

    return count


print(num_subarrays_with_sum([1, 0, 1, 0, 1], 2))   # 4
print(num_subarrays_with_sum([0, 0, 0, 0, 0], 0))   # 15

### Walkthrough: `[1, 0, 1, 0, 1]`, goal=2

```
prefix = {0:1}

r=0, nums[0]=1: total=1  need (1-2)=-1  → prefix.get(-1)=0   count=0  prefix={0:1, 1:1}
r=1, nums[1]=0: total=1  need (1-2)=-1  → 0                   count=0  prefix={0:1, 1:2}
r=2, nums[2]=1: total=2  need (2-2)=0   → prefix.get(0)=1     count=1  prefix={0:1,1:2,2:1}
r=3, nums[3]=0: total=2  need (2-2)=0   → prefix.get(0)=1     count=2  prefix={0:1,1:2,2:2}
r=4, nums[4]=1: total=3  need (3-2)=1   → prefix.get(1)=2     count=4  prefix={0:1,1:2,2:2,3:1}

Result: 4 ✅
```

At r=4, `prefix.get(1) = 2` means there are 2 positions where prefix sum was 1:
- After index 0 (prefix=1): subarray [1,0,1] ✅
- After index 1 (prefix=1): subarray [1,0,1] ✅ (different starting point)

---
## Type 2 Alternative: atMost(k) - atMost(k-1) Trick

For problems where "exactly k" is hard, use:
```
exactly(k) = atMost(k) - atMost(k-1)
```

This works when the sliding window CAN handle `atMost` (no zeros or negatives causing issues).

In [ ]:
def subarrays_with_k_distinct(s, k):
    """
    Count subarrays with EXACTLY k distinct characters.
    
    exactly(k) = atMost(k) - atMost(k-1)
    
    atMost(k): standard sliding window works fine for this.
    """
    from collections import defaultdict

    def at_most(k):
        freq  = defaultdict(int)
        l     = 0
        count = 0

        for r in range(len(s)):
            freq[s[r]] += 1

            while len(freq) > k:
                freq[s[l]] -= 1
                if freq[s[l]] == 0:
                    del freq[s[l]]
                l += 1

            # Every subarray ending at r with left in [l, r] is valid
            # That's (r - l + 1) subarrays
            count += r - l + 1

        return count

    return at_most(k) - at_most(k - 1)


print(subarrays_with_k_distinct("pqpqs", 2))   # 7
print(subarrays_with_k_distinct("aabac", 2))   # 7

### Why does `count += r - l + 1` count subarrays?

```
s = "pqp", at r=2, l=0 (window valid)

Subarrays ENDING at r=2:
  s[2..2] = "p"    ← length 1
  s[1..2] = "qp"   ← length 2
  s[0..2] = "pqp"  ← length 3

Count = r - l + 1 = 2 - 0 + 1 = 3  ✅
```

Every valid left boundary from `l` to `r` gives one valid subarray ending at `r`.

---
## Decision Tree: Which Strategy to Use?

```
Is the question asking for a COUNT?
│
├── NO  → Type 1 (find longest/shortest)
│         Use: standard sliding window with while-shrink
│         Answer: track max/min of (r - l + 1)
│
└── YES → Type 2 (count subarrays)
          │
          ├── Condition uses SUM?
          │   └── Use prefix sum hashmap
          │       seed with {0: 1}
          │       count += prefix.get(total - goal, 0)
          │
          └── Condition uses DISTINCT count / other?
              └── Use atMost(k) - atMost(k-1)
                  count += r - l + 1 inside atMost
```

---
## Side-by-Side Comparison

| | Type 1: Find Longest | Type 2: Count Subarrays |
|---|---|---|
| Question | "What is the max length?" | "How many subarrays?" |
| Windows per r | 1 (the current valid window) | Many (all valid l's) |
| l movement | Shrink until valid | Shrink until valid OR use prefix |
| Answer update | `max(ans, r - l + 1)` | `count += prefix.get(...)` or `count += r - l + 1` |
| Zeros/negatives OK? | Yes (shrink handles it) | Prefix sum needed for sum problems |
| Key seed | None | `prefix = {0: 1}` |


---
## The `prefix = {0: 1}` Explained Once and For All

```
Without seed:   works only if goal-sum subarrays don't start at index 0
With seed:      works always

Example: nums=[1, 1], goal=2

Without {0:1}:
  r=0: total=1, need -1 → 0
  r=1: total=2, need 0  → prefix.get(0) = NOT FOUND → 0  ← WRONG (answer is 1)

With {0:1}:
  r=0: total=1, need -1 → 0
  r=1: total=2, need 0  → prefix.get(0) = 1  ← CORRECT

The seed says: "before the array starts, we've seen prefix sum 0 exactly once."
This lets us count subarrays that start from index 0.
```

---
## Problems You've Solved — Categorized

### 🟢 Type 1 (Find Longest/Shortest)
| Problem | Condition | Shrink when |
|---------|-----------|-------------|
| 3. Longest Substring Without Repeating | No duplicates | duplicate enters |
| 1004. Max Consecutive Ones III | At most k zeros | zero_count > k |
| 424. Longest Repeating Char Replacement | len - maxFreq ≤ k | len - maxFreq > k |

### 🔴 Type 2 (Count Subarrays)
| Problem | Strategy | Key |
|---------|----------|-----|
| 930. Binary Subarrays With Sum | Prefix sum hashmap | `{0:1}` seed |
| Count substrings with K distinct | atMost(k) - atMost(k-1) | `count += r - l + 1` |